In [1]:
import hist
import rich
import gzip
import correctionlib
import correctionlib.convert
import numpy as np
from coffea import util

In [2]:
year = "2023postBPix"

processed_histograms = util.load(f"../outputs/zplusl_os/{year}/{year}_processed_histograms.coffea")

data = processed_histograms["Data"]['probe_lepton'][{"variation":"nominal"}].project("probe_lepton_pt", "is_passing_lepton", "is_barrel_lepton", "category")
wz = processed_histograms["WZ"]['probe_lepton'][{"variation":"nominal"}].project("probe_lepton_pt", "is_passing_lepton", "is_barrel_lepton", "category")

In [3]:
numerator = data[{"is_passing_lepton":True}] + (wz[{"is_passing_lepton":True}] * -1)
numerator

Hist(
  Variable([5, 7, 10, 20, 30, 40, 50, 80], name='probe_lepton_pt', label='$p_T(\\ell)$ [GeV]'),
  IntCategory([0, 1], name='is_barrel_lepton', label='is barrel $\\ell$'),
  StrCategory(['electron', 'muon'], name='category'),
  storage=Weight()) # Sum: WeightedSum(value=11577.1, variance=11688.5)

In [4]:
denominator = data[{"is_passing_lepton":sum}] + (wz[{"is_passing_lepton":sum}] * -1)
denominator

Hist(
  Variable([5, 7, 10, 20, 30, 40, 50, 80], name='probe_lepton_pt', label='$p_T(\\ell)$ [GeV]'),
  IntCategory([0, 1], name='is_barrel_lepton', label='is barrel $\\ell$'),
  StrCategory(['electron', 'muon'], name='category'),
  storage=Weight()) # Sum: WeightedSum(value=199230, variance=199363)

In [5]:
num = numerator.values()
den = denominator.values()
fr = np.where((num > 0) & (den > 0), num / (den + 1e-32), 1)
fr_hist = hist.Hist(*numerator.axes[:], data=fr)
fr_hist

Hist(
  Variable([5, 7, 10, 20, 30, 40, 50, 80], name='probe_lepton_pt', label='$p_T(\\ell)$ [GeV]'),
  IntCategory([0, 1], name='is_barrel_lepton', label='is barrel $\\ell$'),
  StrCategory(['electron', 'muon'], name='category'),
  storage=Double()) # Sum: 3.815931632636275

In [6]:
fr_hist.name = f"os_fake_rate"
fr_hist.label = f"OS FR weight"
os_fr_weight = correctionlib.convert.from_histogram(fr_hist)
os_fr_weight.description = f"OS fake rates"
# set overflow bins behavior (default is to raise an error when out of bounds)
os_fr_weight.data.flow = "clamp"
rich.print(os_fr_weight)

📈 os_fake_rate (v0)
OS fake rates
Node counts: Binning: 1, Category: 21
╭──────────── ▶ input ────────────╮ ╭─────── ▶ input ────────╮ ╭─────── ▶ input ────────╮
│ probe_lepton_pt (real)          │ │ is_barrel_lepton (int) │ │ category (string)      │
│ $p_T(\ell)$ [GeV]               │ │ is barrel $\ell$       │ │ category               │
│ Range: [5.0, 80.0), overflow ok │ │ Values: 0, 1           │ │ Values: electron, muon │
╰─────────────────────────────────╯ ╰────────────────────────╯ ╰────────────────────────╯
╭───── ◀ output ──────╮
│ OS FR weight (real) │
│ No description      │
╰─────────────────────╯

In [7]:
cset = correctionlib.schemav2.CorrectionSet(
    schema_version=2,
    description=f"{year} OS fake rate weight",
    corrections=[
        os_fr_weight
    ],
)
with gzip.open(f"../analysis/data/{year}_os_fr_weight.json.gz", "wt") as fout:
    fout.write(cset.json(exclude_unset=True))